# 02 — Sensor Data Analysis
Analyse simulated IMU signals — noise characteristics, signal structure,
and how sensor readings relate to the underlying physics.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.physics.ball_physics_model import BallPhysicsModel
from src.sensors.sensor_noise_simulator import SensorNoiseSimulator
from src.utils.visualization import plot_imu_signal

model = BallPhysicsModel()
traj  = model.simulate(20.0, 35.0, spin_rpm=1000.0)
print(f'Trajectory: {len(traj.t)} samples, duration {traj.t[-1]:.2f}s')

## 1. Clean vs. noisy IMU signals

In [ ]:
sim_clean = SensorNoiseSimulator(seed=0)
sim_noisy = SensorNoiseSimulator(seed=0)

clean_imu = sim_clean.simulate_imu(traj, noise_level=0.0)
noisy_imu = sim_noisy.simulate_imu(traj, noise_level=1.0)

Path('../outputs/plots').mkdir(parents=True, exist_ok=True)
plot_imu_signal(clean_imu, save_path=Path('../outputs/plots/imu_signal.png'), noisy_imu=noisy_imu)

from IPython.display import Image
Image('../outputs/plots/imu_signal.png', width=750)

## 2. Noise level impact on signal SNR

In [ ]:
noise_levels = [0.0, 0.5, 1.0, 2.0]
fig, axes = plt.subplots(len(noise_levels), 1, figsize=(10, 10), sharex=True)

for ax, nl in zip(axes, noise_levels):
    sim = SensorNoiseSimulator(seed=42)
    imu = sim.simulate_imu(traj, noise_level=nl)
    ax.plot(imu.t, imu.acc_x, lw=0.8, color='steelblue', label=f'acc_x  σ×{nl}')
    ax.set_ylabel(f'noise={nl}', fontsize=9)
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Acc X at Different Noise Levels', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Gyroscope reading vs. true spin rate

In [ ]:
spin_rpms = np.linspace(-2000, 2000, 20)
gyro_means = []
for rpm in spin_rpms:
    t = model.simulate(20.0, 30.0, float(rpm))
    imu = SensorNoiseSimulator(0).simulate_imu(t, noise_level=0.0)
    gyro_means.append(imu.gyro_z.mean())

true_rads = spin_rpms * (2 * np.pi / 60)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(true_rads, gyro_means, 'o-', color='#e15759')
ax.plot(true_rads, true_rads, 'k--', lw=1, label='Perfect')
ax.set_xlabel('True spin (rad/s)')
ax.set_ylabel('Gyro mean (rad/s)')
ax.set_title('Gyroscope Reading vs. True Spin Rate (no noise)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Dataset summary statistics

In [ ]:
import numpy as np
from pathlib import Path

X = np.load('../outputs/dataset_X.npy')
y = np.load('../outputs/dataset_y.npy')

print(f'X shape: {X.shape}  (samples, timesteps, channels)')
print(f'y shape: {y.shape}  (samples, [speed_norm, angle_norm, spin_norm])')
print(f'y min={y.min():.4f}  max={y.max():.4f}  (should be [0,1])')

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
labels = ['Speed norm', 'Angle norm', 'Spin norm']
for ax, i, lbl in zip(axes, range(3), labels):
    ax.hist(y[:, i], bins=40, color='steelblue', edgecolor='white', lw=0.3)
    ax.set_title(lbl)
    ax.set_xlabel('Normalised value')
fig.suptitle('Dataset Label Distributions', fontsize=12)
plt.tight_layout()
plt.show()